# k-Nearest Neighbours, Support Vector Machines and Naive Bayes – Titanic

**Dataset:** Titanic – 891 passengers (Kaggle), a classic classification example. File: `../00_Data/titanic.csv`.

**Question:** Can the survival of a Titanic passenger be predicted from a few characteristics with distance-based
methods (k-NN, SVM) and with Naive Bayes?

**After this exercise you can ...**
- apply k-NN, explain why features must be scaled and choose $k$ by cross-validation,
- explain the role of the margin, the support vectors, $C$ and the kernel of an SVM,
- compute a Naive Bayes classification by hand and apply Naive Bayes to categorical data.

## Concept

The concepts behind this exercise (formulas, worked examples and additional explanations) are
explained in the notebook [knn_naive_bayes_svm_concept.ipynb](knn_naive_bayes_svm_concept.ipynb). Read it before you start.

## Libraries and settings

In [ ]:
# Libraries
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import CategoricalNB
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import ConfusionMatrixDisplay

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Show current working directory
print(os.getcwd())

## Exercise 1: k-NN and Naive Bayes by hand

**a) k-NN with and without scaling.** A new passenger is 30 years old and paid a fare of 40 pounds. Five known
passengers:

| passenger | survived? | age [years] | fare [£] |
|---|---|---|---|
| P1 | no | 58 | 41 |
| P2 | yes | 29 | 70 |
| P3 | yes | 33 | 75 |
| P4 | no | 62 | 38 |
| P5 | no | 31 | 8 |

1. Compute the Euclidean distances on the **original scale** and classify the new passenger with $k=3$.
2. Standardise the differences with the standard deviations of the whole data set (age: 13 years, fare: 50 £), i.e.
   use $\Delta z = \Delta x / s$, and classify again with $k=3$.
3. Why do the results differ?

**b) Naive Bayes.** Of the 891 passengers, 342 survived and 549 died. Among the survivors there were 233 women and 136
first-class passengers; among those who died there were 81 women and 80 first-class passengers. Compute the Naive Bayes
scores for a **woman in 1st class** and her probability of survival. Compare
with the observed survival rate of women in 1st class (91 of 94).

**c)** Verify your results with Python.

In [ ]:
# a) k-NN with and without scaling
new = np.array([30, 40])
X_known = np.array([[58, 41], [29, 70], [33, 75], [62, 38], [31, 8]])
survived = np.array(['no', 'yes', 'yes', 'no', 'no'])
sd = np.array([13, 50])
# Your code here

# b) Naive Bayes
# Your code here

**Your answer:** ...

## Exercise 2: Prepare the Titanic data

a) Import the Titanic data and prepare the features `male`, `Age` (missing values replaced by the median), `Pclass`,
`SibSp`, `Parch` and `Fare`. Split the data into a stratified training (80 %) and test set (20 %) with `random_state=42`.

b) Create a scatter plot of `Age` against `Fare` (logarithmic y-axis), coloured by `Survived`, and compare the scales of
the features.

In [ ]:
# a) Import and prepare the data
titanic = pd.read_csv('../00_Data/titanic.csv', sep=',')
features = ['male', 'Age', 'Pclass', 'SibSp', 'Parch', 'Fare']
# Your code here
X_train, X_test, y_train, y_test = ___

# b) Scatter plot Age vs. Fare
# Your code here

**Your answer:** ...

## Exercise 3: k-NN – the effect of scaling and the choice of k

a) Compute the 5-fold CV accuracy (on the training set) of `KNeighborsClassifier(n_neighbors=5)` on the **unscaled**
features and of `make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))`. Explain the difference.

b) Compute the 5-fold CV accuracy of the scaled k-NN for $k = 1, 2, \dots, 40$ and plot it against $k$. Choose $k$ and
compute accuracy and confusion matrix on the test set.

c) Predict the survival probability of a 30-year-old man in 3rd class without relatives (fare 8 £) and of a
30-year-old woman in 1st class without relatives (fare 80 £).

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# a) Unscaled vs. scaled
# Your code here

# b) Choice of k
ks = range(1, 41)
# Your code here

# c) Two new passengers
new_passengers = pd.DataFrame({'male': [1, 0], 'Age': [30, 30], 'Pclass': [3, 1],
                               'SibSp': [0, 0], 'Parch': [0, 0], 'Fare': [8, 80]})
# Your code here

**Your answer:** ...

## Exercise 4: Support vector machines – C and kernel

a) Compute the 5-fold CV accuracy and the number of support vectors of
`make_pipeline(StandardScaler(), SVC(kernel=..., C=...))` for `kernel` ∈ {`'linear'`, `'rbf'`} and
$C$ ∈ {0.01, 1, 100} on the training set (all six features).

b) To *see* the boundaries, fit the same six models on only the two features `Age` and `log_fare = log(1 + Fare)` and
plot the decision regions in a 2 × 3 grid (the code for the grid is given), with the support vectors circled.

c) Describe the influence of $C$ and of the kernel. Which model would you choose?

In [ ]:
# a) CV accuracy and number of support vectors (all features)
# Your code here

# b) Decision regions with two features
X2_train = pd.DataFrame({'Age': X_train['Age'], 'log_fare': np.log1p(X_train['Fare'])})
xx, yy = np.meshgrid(np.linspace(0, 82, 300), np.linspace(0, 6.5, 300))
grid = pd.DataFrame({'Age': xx.ravel(), 'log_fare': yy.ravel()})

fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharex=True, sharey=True)
for row, kernel in enumerate(['linear', 'rbf']):
    for col, C in enumerate([0.01, 1, 100]):
        ax = axes[row, col]
        svm = ___
        zz = svm.predict(grid).reshape(xx.shape)
        ax.contourf(xx, yy, zz, levels=[-0.5, 0.5, 1.5], colors=['tab:red', 'tab:blue'], alpha=0.15)
        # Plot the training points (coloured by Survived) and circle the support vectors (svm[-1].support_)
        # Your code here
plt.show()

**Your answer:** ...

## Exercise 5: Naive Bayes with categorical features

a) Fit a `CategoricalNB(alpha=1e-10)` (practically no smoothing) on the two features `male` and `Pclass` of **all**
passengers and predict the survival probability of a woman in 1st class. Compare with your hand calculation of
Exercise 1b.

b) Compute the 5-fold CV accuracy of a `CategoricalNB()` with the features `male`, `Pclass` and `age_group`
(0 = child under 10, 1 = 10–59 years, 2 = 60 years and older) on the training set. Compare with k-NN and SVM.

In [ ]:
# a) CategoricalNB on male and Pclass (all passengers)
X_cat = pd.DataFrame({'male': titanic['male'], 'Pclass': titanic['Pclass'] - 1})   # categories must start at 0
# Your code here

# b) CategoricalNB with age group
# Your code here

**Your answer:** ...

## Exercise 6: Reflection

Compare k-NN, SVM and Naive Bayes: for which of the three methods is

a) feature scaling necessary,

b) the training fast but the prediction slow,

c) a very large number of features (e.g. thousands of words) unproblematic?

**Your answer:** ...

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')